In [ ]:
import ee
import os
import json
import requests
from datetime import date
from dotenv import load_dotenv
from collections import defaultdict

In [ ]:
load_dotenv()
PROJECT_ID = os.getenv("PROJECT_ID")
SERVICE_ACCOUNT = os.getenv("SERVICE_ACCOUNT")
AOI_PATH = "../data/AOIs/aoi.geojson"
OUTPUT_DIR = "../data/satellite_images"

In [ ]:
def init_ee():
    if PROJECT_ID is None or SERVICE_ACCOUNT is None:
        raise ValueError(
            "PROJECT_ID and SERVICE_ACCOUNT must be set in environment variables."
        )
    credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, "../geopulse-key.json")
    ee.Initialize(credentials, project=PROJECT_ID)
    print(f"✅ Earth Engine initialized with project: {PROJECT_ID}")
    
init_ee()

In [ ]:
def mask_s2_clouds(image):
    scl = image.select("SCL")
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask).copyProperties(image, ["system:time_start"])

In [ ]:
def get_geometry(aoi_geojson):
    if aoi_geojson["type"] == "FeatureCollection":
        geom = ee.Geometry(aoi_geojson["features"][0]["geometry"])
    elif aoi_geojson["type"] == "Feature":
        geom = ee.Geometry(aoi_geojson["geometry"])
    else:
        raise ValueError(
            "Unsupported GeoJSON type. Must be Feature or FeatureCollection."
        )
    return geom.simplify(100)

In [ ]:
def setinel_2(start_date, end_date, cloud_coverage=20, geom=None):
    dataset = "COPERNICUS/S2_SR_HARMONIZED"
    bands = ["B2", "B3", "B4", "B8", "B11", "B12"]
    scale_map = {"B2": 10, "B3": 10, "B4": 10, "B8": 10, "B11": 20, "B12": 20}
    col = (
        ee.ImageCollection(dataset)
        .filterBounds(geom)
        .filterDate(str(start_date), str(end_date))
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_coverage))
        .map(mask_s2_clouds)
    )
    return col.select(bands), scale_map

In [ ]:
def landsat_8(start_date, end_date, cloud_coverage=20, geom=None):
    dataset = "LANDSAT/LC08/C02/T1_L2"
    bands = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]
    scale_map = {b: 30 for b in bands}
    col = (
        ee.ImageCollection(dataset)
        .filterBounds(geom)
        .filterDate(str(start_date), str(end_date))
        .filter(ee.Filter.lt("CLOUD_COVER", cloud_coverage))
    )
    return col.select(bands), scale_map

In [ ]:
def save_image_band(img, band, scale, geom, folder_path, prefix):
    img = img.clip(geom)  # clip to AOI
    date_str = img.date().format("YYYYMMdd").getInfo()
    filename = f"{prefix}_{band}_{date_str}.TIF"
    filepath = os.path.join(folder_path, filename)
    if os.path.exists(filepath):
        return
    download_url = img.select(band).getDownloadURL(
        {
            "scale": scale,
            "region": geom.bounds().getInfo()["coordinates"],
            "format": "GEO_TIFF",
        }
    )
    response = requests.get(download_url)
    if response.status_code == 200:
        with open(filepath, "wb") as f:
            f.write(response.content)
    else:
        print(f"❌ Failed: {filename}")

In [ ]:
def download_satellite_data(aoi_geojson, start_date, end_date, satellite="Sentinel-2", cloud_coverage=20):
    geom = get_geometry(aoi_geojson)
    satellite = satellite.lower()

    if satellite in ["sentinel-2", "s2"]:
        col, scale_map = setinel_2(start_date, end_date, cloud_coverage, geom)
        prefix = "S2"
    elif satellite in ["landsat-8", "landsat8"]:
        col, scale_map = landsat_8(start_date, end_date, cloud_coverage, geom)
        prefix = "L8"
    else:
        raise ValueError(
            "Unsupported satellite. Choose 'Sentinel-2', 'Landsat-8', or 'MODIS'."
        )

    count = col.size().getInfo()
    if count == 0:
        print(f"⚠️ No {satellite} images found.")
        return

    print(f"📦 Found {count} {satellite} images.")
    saved_months = set()
    images_by_month = defaultdict(list)
    
    # Organize images by year and month
    for i in range(count):
        img = ee.Image(col.toList(count).get(i))
        year = img.date().format("YYYY").getInfo()
        month = img.date().format("MMMM").getInfo()
        # Skip if this month already has a saved image
        if (year, month) in saved_months:
            continue
        saved_months.add((year, month))
        # Add image to the monthly list
        images_by_month[(year, month)].append(img)

    print(f"🗂️ Organized images into {len(images_by_month)} months for download.")
    # Download images month by month
    for (year, month), images in sorted(images_by_month.items()):
        folder_path = os.path.join(OUTPUT_DIR, satellite.replace(" ", "_"), year, month)
        os.makedirs(folder_path, exist_ok=True)
        # Download all images and bands for this month
        for img in images:
            for band, scale in scale_map.items():
                save_image_band(img, band, scale, geom, folder_path, prefix)

        # Month-level summary
        print(f"✅ Completed downloads for {month} {year}: {folder_path}")
    print(f"🎯 All downloads completed. Files saved in: {os.path.abspath(OUTPUT_DIR)}")

In [ ]:
satellite = "sentinel-2"
aoi_geojson = json.load(open(AOI_PATH))
start_date = date(2020, 1, 1)
end_date = date(2025, 8, 30)
download_satellite_data(aoi_geojson, start_date, end_date, satellite=satellite)